## Setup

Runs once per Colab session. Clone repo, install deps, mount Google Drive, link dataset.

In [ ]:
import os
import subprocess
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ── 1. Clone / pull repo ──
REPO_DIR = "/content/SymbioPan"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "https://github.com/hoangtung386/SymbioPan", REPO_DIR], check=True)
    print("Cloned SymbioPan")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    print("Updated SymbioPan")

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

In [ ]:
# ── 2. Install dependencies ──
subprocess.run([sys.executable, "-m", "pip", "install", "-r", f"{REPO_DIR}/requirements.txt", "-q"], check=True)
print("Dependencies installed")

In [ ]:
# ── 3. Login HuggingFace for gated Virchow2 ──
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"), add_to_git_credential=True)
print("HuggingFace login OK")

In [ ]:
# ── 4. Mount Google Drive & link dataset ──
#    Upload dataset_processed/ to your Drive first.
#    Edit DRIVE_DATASET_PATH to match your folder.
from google.colab import drive

drive.mount("/content/drive")

DRIVE_DATASET_PATH = "/content/drive/MyDrive/PUMA/dataset_processed"
if os.path.exists(DRIVE_DATASET_PATH):
    if os.path.islink(f"{REPO_DIR}/dataset_processed"):
        os.unlink(f"{REPO_DIR}/dataset_processed")
    os.symlink(DRIVE_DATASET_PATH, f"{REPO_DIR}/dataset_processed")
    print(f"Dataset linked from {DRIVE_DATASET_PATH}")
else:
    print(f"WARNING: Dataset not found at {DRIVE_DATASET_PATH}")
    print("Please upload dataset_processed/ to your Drive and update DRIVE_DATASET_PATH")

print("\nSetup complete.")

## 0. Data Preprocessing

Convert raw TIFF ROIs + GeoJSON annotations → `dataset_processed/` (`.npy` files with rare-augmented crops).
All modules in `data/preprocessing/` are used here: `geojson_parser`, `flow_generator`, `preprocess`.

In [ ]:
# ── GeoJSON Parsing Demo ──
import matplotlib.pyplot as plt
import numpy as np

from data.preprocessing.flow_generator import compute_hv_map
from data.preprocessing.geojson_parser import parse_geojson_masks

print("GeoJSON parser:", parse_geojson_masks.__name__)
print("HV map:", compute_hv_map.__name__)

# Quick demo: show HV map on a tiny mock instance mask
mock_inst = np.zeros((64, 64), dtype=np.int32)
mock_inst[20:40, 20:40] = 1
mock_inst[35:55, 35:55] = 2
hv = compute_hv_map(mock_inst)
print(f"HV map shape: {hv.shape}  dtype: {hv.dtype}")

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(mock_inst, cmap="tab10")
axes[0].set_title("Instance mask")
axes[1].imshow(hv[0], cmap="RdBu_r")
axes[1].set_title("HV-X")
axes[2].imshow(hv[1], cmap="RdBu_r")
axes[2].set_title("HV-Y")
plt.tight_layout(); plt.show()
print("Preprocessing modules OK.")

### Run Full Preprocessing Pipeline

Reads `Dataset/01_training_dataset_tif_ROIs/*.tif` + GeoJSON → writes `dataset_processed/`.
Skip this cell if `dataset_processed/` is already prepared (e.g. from Drive).

In [ ]:
from configs import PATHS
from data.preprocessing import main as preprocess_main

if Path(str(PATHS.raw_dir)).exists() and any(Path(str(PATHS.raw_dir)).iterdir()):
    print("Raw dataset found. Running preprocessing...")
    preprocess_main()
    print("Preprocessing complete.")
else:
    print(f"Raw data not found at {PATHS.raw_dir}")
    print("Skipping preprocessing. Make sure dataset_processed/ is available from Drive.")

# SymbioPan v8 — CellPath Training

**Architecture**: Virchow2 ViT-H/14 encoder → ConvNeXt-Tiny CNN backbone → HierarchicalFPN → ParallelDecoders (DeepLabV3+ tissue, CellViT++ nuclei, HoVerNeXt NP/HV, BoundaryAttention)

**Preprocessed data** ready at `dataset_processed/` (see Section 0 above to regenerate).

In [ ]:
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import torch

warnings.filterwarnings("ignore")
print(f"Python {sys.version}\nPyTorch {torch.__version__}  CUDA {torch.version.cuda}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# GPU Optimisation (H200 / H100 Tensor Cores)
import os

if device.type == "cuda":
    # ── GPU info ──
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {p.name} | {p.total_memory/1e9:.1f} GB | CC {p.major}.{p.minor}")

    # ── TensorFloat-32 (TF32) on Tensor Cores ──
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
    print(f"  TF32 enabled | matmul_precision=high")

    # ── cuDNN auto-tuner ──
    torch.backends.cudnn.benchmark = True

    # ── BF16 AMP (native on H100/H200) ──
    bf16_avail = torch.cuda.is_bf16_supported()
    print(f"  BF16 support: {bf16_avail}")

    # ── CPU threads for DataLoader ──
    cpu_cnt = os.cpu_count() or 4
    print(f"  CPU cores: {cpu_cnt}")
    torch.set_num_threads(min(4, cpu_cnt))

    # ── Expandable memory segments (reduces fragmentation) ──
    os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

else:
    print("CPU mode — skipping GPU optimizations")

---
## 1. Configuration

All defaults live in `configs/defaults.py`. You can override via CLI or modify `Stage1Config` directly.

In [ ]:
from configs import PATHS
from configs import STAGE1_DEFAULT_CONFIG as C
from configs.defaults import linear_ramp
from training.gpu_setup import detect_gpu_setup

gpu_info = detect_gpu_setup()
print(f"GPU: {gpu_info}\n")
print(f"Config:\n  epochs={gpu_info.epochs}  lr={gpu_info.lr}  batch_size={gpu_info.batch_size}")
print(f"  image_size={gpu_info.image_size}  warmup_epochs={gpu_info.warmup_epochs}")
print(f"  num_workers={gpu_info.num_workers}  compile_model={gpu_info.compile_model}")
print(f"  use_context_encoder={gpu_info.use_context_encoder}  use_stain_aug={gpu_info.use_stain_aug}")
print(f"  fine_tune_last_n_blocks={gpu_info.fine_tune_last_n_blocks}")
print(f"  Data: {PATHS.data_dir}")

---
## 2. Data Constants & Dataset

In [ ]:
from data.constants import (
    INTERNAL_TISSUE_ID_TO_NAME,
    NUCLEI_CLASS_WEIGHTS,
    NUM_NUCLEI_CLASSES,
    NUM_TISSUE_CLASSES,
    PUMA_NUCLEI_ID_TO_NAME,
    RARE_NUCLEI_IDS,
    RARE_TISSUE_IDS,
    TISSUE_CLASS_WEIGHTS,
)

print(f"Tissue classes ({NUM_TISSUE_CLASSES}): {dict(INTERNAL_TISSUE_ID_TO_NAME)}")
print(f"Nuclei classes  ({NUM_NUCLEI_CLASSES}): {dict(PUMA_NUCLEI_ID_TO_NAME)}")
print(f"Rare tissue IDs: {RARE_TISSUE_IDS}")
print(f"Rare nuclei IDs: {RARE_NUCLEI_IDS}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].bar(range(len(TISSUE_CLASS_WEIGHTS)), TISSUE_CLASS_WEIGHTS, color="tab:blue")
axes[0].set_title("Tissue class weights")
axes[1].bar(range(len(NUCLEI_CLASS_WEIGHTS)), NUCLEI_CLASS_WEIGHTS, color="tab:orange")
axes[1].set_title("Nuclei class weights")
plt.tight_layout(); plt.show()

### Dataset instantiation + transforms

In [ ]:
from data.dataset import PUMADataset, get_train_transforms, get_val_transforms

# Quick demo: instantiate dataset and inspect a sample
val_tfms = get_val_transforms(C.image_size)
train_tfms = get_train_transforms(C.image_size, use_stain_aug=C.use_stain_aug)

ds = PUMADataset(
    data_dir=str(PATHS.data_dir),
    transforms=train_tfms,
    use_context=C.use_context_encoder,
)
print(f"Dataset size: {len(ds)}")
sample = ds[0]
print(f"Keys: {list(sample.keys())}")
print(f"Image: {sample['image'].shape}  tissue: {sample['tissue_sem'].shape}")
print(f"nuclei_np: {sample['nuclei_np'].shape}  nc: {sample['nuclei_nc'].shape}  hv: {sample['nuclei_hv'].shape}")
print(f"site_id: {sample['site_id']}  base_name: {sample['base_name']}")
if C.use_context_encoder:
    print(f"context_roi: {sample.get('context_roi', 'N/A')}")

In [ ]:
# Visualise a few training samples
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
inv_mean = [-m/s for m,s in zip([0.485,0.456,0.406], [0.229,0.224,0.225])]
inv_std  = [1/s for s in [0.229,0.224,0.225]]
for i in range(2):
    s = ds[i]
    img = s["image"].permute(1,2,0).numpy()
    img = img * inv_std + inv_mean
    axes[i,0].imshow(np.clip(img, 0, 1))
    axes[i,0].set_title(f"Image {i}")
    axes[i,1].imshow(s["tissue_sem"], cmap="tab10", vmin=0, vmax=4)
    axes[i,1].set_title("Tissue")
    axes[i,2].imshow(s["nuclei_nc"], cmap="tab10", vmin=0, vmax=9)
    axes[i,2].set_title("Nuclei class")
    axes[i,3].imshow(s["nuclei_hv"][0], cmap="RdBu_r")
    axes[i,3].set_title("HV-X")
plt.tight_layout(); plt.show()

---
## 3. Model Architecture

In [ ]:
# ── Full model import (covers every file in models/) ──
from models import UnifiedPanopticNet, build_cnn_backbone
from models.components import ContextEncoder, ContextFusionModule
from models.decoders import (
    ParallelDecoders,
)
from models.fpn_aggregator import HierarchicalFPN

print("models/__init__.py           ─ UnifiedPanopticNet, build_cnn_backbone")
print("models/backbone.py           ─ ConvNeXt-Tiny")
print("models/encoder.py            ─ UnifiedPanopticEncoder, build_virchow2_vit, extract_intermediate_features")
print("models/cross_attention.py    ─ SpatialInjector")
print("models/fpn_aggregator.py     ─ HierarchicalFPN")
print("models/decoders.py           ─ ParallelDecoders, MutualFeatureExchange, ASPP,")
print("                               DeepLabV3PlusTissueHead, CellViTPlusPlusNucleiDecoder, HoVerNeXtNucleiHead")
print("models/panoptic_net.py       ─ UnifiedPanopticNet (master)")
print("models/components/           ─ BoundaryAttentionModule, ContextEncoder, ContextFusionModule")
print()

# Build CNN backbone (ConvNeXt-Tiny, no pretrained weights needed for demo)
cnn = build_cnn_backbone(pretrained=False)
print(f"CNN backbone: ConvNeXt-Tiny  feature_dims={cnn.feature_info.channels()}")

# Full model (with placeholder encoder — won't load Virchow2 in this demo)
# If HuggingFace is unreachable, build what we can for architecture demo.
try:
    model = UnifiedPanopticNet(
        cnn_model=cnn,
        load_encoder_weights=False,
        use_context_encoder=gpu_info.use_context_encoder,
    )
    print(f"Model param count: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

    # ── torch.compile for H200 Tensor Cores ──
    if device.type == "cuda" and gpu_info.compile_model:
        try:
            model = torch.compile(model, mode="reduce-overhead", fullgraph=False)
            print("  Model compiled with torch.compile (reduce-overhead)")
        except Exception as e:
            print(f"  torch.compile skipped: {e}")
except Exception as e:
    print(f"Warning: Could not build full model (Virchow2 unreachable): {e}")
    print("Proceeding with component-level demos below.")
    model = None

In [ ]:
# Forward pass on dummy data
if model is not None:
    B = 2
    dummy_img = torch.randn(B, 3, gpu_info.image_size, gpu_info.image_size)
    dummy_ctx = torch.randn(B, 3, gpu_info.context_roi_size, gpu_info.context_roi_size) if gpu_info.use_context_encoder else None
    with torch.no_grad():
        try:
            out = model(dummy_img, context_roi=dummy_ctx)
        except Exception as e:
            print(f"Forward pass failed (likely torch.compile dynamo): {e}")
            print("Falling back to uncompiled forward pass...")
            if hasattr(model, '_orig_mod'):
                out = model._orig_mod(dummy_img, context_roi=dummy_ctx)
            else:
                raise
    for k, v in out.items():
        print(f"{k:12s}: {list(v.shape)}")
else:
    print("Skipping forward pass (model not built — see warning above).")

### Individual component inspection

In [ ]:
fpn = HierarchicalFPN(vit_dim=1280, cnn_dims=[96,192,384,768], fpn_dim=256)
vit_tokens = torch.randn(1, 257, 1280)
cnn_feats = [torch.randn(1, d, 256//(2**i), 256//(2**i)) for i, d in enumerate([96,192,384,768])]
vit_inter = torch.randn(4, 1, 1280, 64, 64)
fpn_out, low_feat = fpn(vit_tokens, cnn_feats, vit_inter, img_size=256)
print(f"FPN outputs: { {k: list(v.shape) for k,v in fpn_out.items()} }")
print(f"Low-level feat: {list(low_feat.shape)}")

In [ ]:
decoders = ParallelDecoders(fpn_dim=256, num_tissue=5, num_nuclei=10, low_level_channels=96)
with torch.no_grad():
    tissue, np, nc, hv, boundary = decoders(fpn_out, low_feat, vit_inter)
print(f"tissue: {list(tissue.shape)}   np: {list(np.shape)}   nc: {list(nc.shape)}")
print(f"hv: {list(hv.shape)}   boundary: {list(boundary.shape)}")

In [ ]:
# Context encoder + fusion demo
if C.use_context_encoder:
    ctx_enc = ContextEncoder(output_dim=256, output_mode="global")
    ctx_fus = ContextFusionModule(context_dim=256, fpn_dim=256)
    dummy_ctx = torch.randn(1, 3, C.context_roi_size, C.context_roi_size)
    ctx_desc = ctx_enc(dummy_ctx)
    print(f"Context descriptor: {list(ctx_desc.shape)}")
    fused = ctx_fus(fpn_out, ctx_desc)
    print(f"Fused FPN p1: {list(fused['p1'].shape)}")

---
## 4. Loss & Metrics

In [ ]:
from utils import MultiTaskUncertaintyLoss, PUMAMetrics

criterion = MultiTaskUncertaintyLoss(
    tissue_weights=torch.tensor(TISSUE_CLASS_WEIGHTS, dtype=torch.float32),
    nuclei_weights=torch.tensor(NUCLEI_CLASS_WEIGHTS, dtype=torch.float32),
)
metrics = PUMAMetrics()

# Use dummy predictions if model forward pass was skipped
try:
    dummy_out = out
except NameError:
    dummy_out = {
        "tissue": torch.randn(2, 5, C.image_size, C.image_size),
        "np": torch.randn(2, 1, C.image_size, C.image_size),
        "nc": torch.randn(2, 10, C.image_size, C.image_size),
        "hv": torch.randn(2, 2, C.image_size, C.image_size),
        "boundary": torch.randn(2, 1, C.image_size, C.image_size),
    }
    B = 2

# Demo forward+loss on dummy batch
loss, branch_losses = criterion(dummy_out, {
    "tissue_sem": torch.randint(0, 5, (B, C.image_size, C.image_size)),
    "nuclei_np": torch.randint(0, 2, (B, C.image_size, C.image_size)),
    "nuclei_nc": torch.randint(0, 10, (B, C.image_size, C.image_size)),
    "nuclei_hv": torch.randn(B, 2, C.image_size, C.image_size),
})
print(f"Total loss: {loss.item():.4f}")
print(f"Branch losses: tissue={branch_losses[0]:.4f}  np={branch_losses[1]:.4f}  nc={branch_losses[2]:.4f}  hv={branch_losses[3]:.4f}")

In [ ]:
# Metrics demo
try:
    dummy_out
except NameError:
    dummy_out = {
        "tissue": torch.randn(2, 5, C.image_size, C.image_size),
        "np": torch.randn(2, 1, C.image_size, C.image_size),
        "nc": torch.randn(2, 10, C.image_size, C.image_size),
        "hv": torch.randn(2, 2, C.image_size, C.image_size),
        "boundary": torch.randn(2, 1, C.image_size, C.image_size),
    }
val_metrics = metrics.calculate_all_metrics(dummy_out, {
    "tissue_sem": torch.randint(0, 5, (B, C.image_size, C.image_size)),
    "nuclei_np": torch.randint(0, 2, (B, C.image_size, C.image_size)),
    "nuclei_nc": torch.randint(0, 10, (B, C.image_size, C.image_size)),
    "nuclei_hv": torch.randn(B, 2, C.image_size, C.image_size),
})
for k, v in val_metrics.items():
    if isinstance(v, float):
        print(f"  {k:25s}: {v:.4f}")

---
## 5. Scheduler & Schedule Visualization

In [ ]:
from utils.scheduler_utils import build_warmup_cosine_scheduler

if model is not None:
    optim_params = model.parameters()
else:
    optim_params = [torch.nn.Parameter(torch.randn(1))]  # dummy for scheduler demo
optimizer = torch.optim.AdamW(optim_params, lr=C.lr, weight_decay=C.weight_decay)
scheduler = build_warmup_cosine_scheduler(optimizer, warmup_epochs=C.warmup_epochs, total_epochs=C.epochs, steps_per_epoch=10)

# Simulate LR schedule
lrs = []
for ep in range(1, C.epochs + 1):
    for _ in range(10):  # simulate steps per epoch
        lrs.append(optimizer.param_groups[0]["lr"])
        scheduler.step()

plt.figure(figsize=(8, 3))
plt.plot(lrs)
plt.axvline(C.warmup_epochs * 10, color="r", ls=":", label=f"warmup end (ep {C.warmup_epochs})")
plt.xlabel("Step"); plt.ylabel("LR"); plt.title("Warm-up Cosine Schedule"); plt.legend(); plt.show()

# Reset scheduler
scheduler = build_warmup_cosine_scheduler(optimizer, warmup_epochs=C.warmup_epochs, total_epochs=C.epochs, steps_per_epoch=10)

In [ ]:
# Focal + SC-DFA ramp schedule
epochs = C.epochs
focal_w = [linear_ramp(e, C.focal_start_epoch, C.focal_full_epoch, C.focal_max_weight) for e in range(epochs)]
scdfa_w = [linear_ramp(e, C.sc_dfa_start_epoch, C.sc_dfa_full_epoch, C.sc_dfa_max_weight) for e in range(epochs)]

plt.figure(figsize=(8, 3))
plt.plot(focal_w, label="Focal-Tversky weight", lw=2)
plt.plot(scdfa_w, label="SC-DFA lambda", lw=2)
plt.axvline(C.focal_start_epoch, color="g", ls=":")
plt.axvline(C.sc_dfa_start_epoch, color="m", ls=":")
plt.xlabel("Epoch"); plt.ylabel("Weight"); plt.title("Smooth Schedule"); plt.legend(); plt.show()

---
## 6. Training Loop (Simulated)

Shows loss curves and metric tracking without running a full training.

In [ ]:
import math

import numpy as np

np.random.seed(42)

num_epochs = 30
history = {
    "train_loss": [], "val_loss": [],
    "tissue_dice": [], "nuclei_dice": [], "rare_tissue_dice": [], "rare_nuclei_dice": [],
}

for ep in range(1, num_epochs + 1):
    decay = math.exp(-0.05 * ep)
    noise = 0.02 * np.random.randn()
    base = 2.5 * decay + 0.1 + noise
    history["train_loss"].append(base + 0.05 * np.random.randn())
    history["val_loss"].append(base + 0.03 * np.random.randn())
    progress = min(ep / 15, 1.0)
    history["tissue_dice"].append(0.6 * progress + 0.3 + 0.02 * np.random.randn())
    history["nuclei_dice"].append(0.4 * progress + 0.2 + 0.02 * np.random.randn())
    history["rare_tissue_dice"].append(0.3 * progress + 0.1 + 0.03 * np.random.randn())
    history["rare_nuclei_dice"].append(0.2 * progress + 0.05 + 0.03 * np.random.randn())

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(history["train_loss"], label="Train")
axes[0].plot(history["val_loss"], label="Val")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend(); axes[0].set_title("Loss")
for k in ["tissue_dice", "nuclei_dice", "rare_tissue_dice", "rare_nuclei_dice"]:
    axes[1].plot(history[k], label=k)
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Dice"); axes[1].legend(fontsize=8); axes[1].set_title("Validation Dice")
plt.tight_layout(); plt.show()

---
## 7. Trainer & CLI Entry Point

The actual training script is `training/stage1_trainer.py`. To train:

```bash
# Minimal run
python scripts/run_stage1.py --batch-size 1 --use-stain-aug --use-context-encoder

# Full training with custom params
python scripts/run_stage1.py --epochs 50 --lr 1e-4 --batch-size 1 --use-stain-aug
```

The trainer will:
- Load/save train/val split (`make_or_load_group_split`)
- Build data loaders with weighted sampling for rare classes
- Apply smooth schedule (Focal-Tversky ramp + SC-DFA ramp)
- Log metrics via `PUMAMetrics`
- Save best checkpoint by selection score

In [ ]:
from training import extract_state_dict, train_one_epoch, validate
from training.gpu_setup import cleanup_gpu_cache
from training.stage1_trainer import apply_smooth_schedule, save_checkpoint

# Check that train_one_epoch and validate are importable
print(f"train_one_epoch: {train_one_epoch.__name__}")
print(f"validate: {validate.__name__}")
print(f"apply_smooth_schedule: {apply_smooth_schedule.__name__}")
print(f"save_checkpoint: {save_checkpoint.__name__}")
print("\nAll training primitives import OK.")

---
## Training & Model Export to Drive

### A. Create held-out test set (3-way split)

Split by source image group to prevent data leakage. Test set is **never** used during training or checkpoint selection.

In [ ]:
import math
import os
import shutil
from pathlib import Path

from torch.utils.data import DataLoader, Subset

from data.dataset import PUMADataset, get_val_transforms
from utils.split_utils import make_or_load_group_split_with_test

# Reuse the same val transforms for test (no augmentation)
val_tfms = get_val_transforms(C.image_size)
context_dir = Path(str(PATHS.raw_dir)) / "01_training_dataset_tif_context_ROIs" if C.use_context_encoder else None

full_ds = PUMADataset(
    data_dir=str(PATHS.data_dir),
    transforms=val_tfms,
    context_dir=context_dir,
    use_context=C.use_context_encoder,
)
split_meta = full_ds.get_split_metadata()

# 3-way split by source group: 70% train, 15% val, 15% test
# The split file is saved separately from the 2-way one to avoid collisions
split_path_3way = str(PATHS.split_file).replace(".npz", "_3way.npz")
train_idx, val_idx, test_idx = make_or_load_group_split_with_test(
    source_names=split_meta["source_names"],
    is_original=split_meta["is_original"],
    split_path=split_path_3way,
    seed=C.seed,
    train_fraction=0.7,
    val_fraction=0.15,
    force_new=True,
    val_original_only=True,
)

test_loader = DataLoader(
    Subset(full_ds, test_idx),
    batch_size=gpu_info.batch_size,
    shuffle=False,
    num_workers=gpu_info.num_workers,
    pin_memory=True,
    persistent_workers=gpu_info.num_workers > 0,
    prefetch_factor=4 if gpu_info.num_workers > 0 else 2,
)

print(f"Train: {len(train_idx)} samples   Val: {len(val_idx)}   Test: {len(test_idx)} (held out)")

### B. Run Training

The trainer uses `make_or_load_group_split` internally (2-way split). The test set created above is only used **after** training for final evaluation.

In [ ]:
from training.stage1_trainer import main as train_main

print("=" * 60)
print("Starting Stage 1 training...")
print("=" * 60)
result = train_main(gpu_info, test_loader=test_loader)
print("\nTraining finished!")

# ── Export entity checkpoints to Google Drive ──
DRIVE_MODEL_DIR = "/content/drive/MyDrive/SymbioPan/models"
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)

for fname in ["puma_epoch_best_s1.pth", "puma_epoch_last_s1.pth"]:
    src = Path(f"checkpoints/{fname}")
    if src.exists():
        shutil.copy(src, f"{DRIVE_MODEL_DIR}/{fname}")
        print(f"\u2705 {fname}  \u2192 {DRIVE_MODEL_DIR}/{fname}")

print("\n" + "=" * 60)
print(" All entity models saved to Google Drive!")
print(f"    {DRIVE_MODEL_DIR}")
print("=" * 60)

In [ ]:
# ── Free GPU memory after training, before test ──
import gc
gc.collect()
torch.cuda.empty_cache()
print("GPU cache cleared before test evaluation")

### C. Training History

Loss and Dice curves per epoch.

In [ ]:
history = result["history"]
epochs = [h["epoch"] for h in history]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss
axes[0].plot(epochs, [h["train_loss"] for h in history], label="Train", lw=1.5)
axes[0].plot(epochs, [h["val_loss"] for h in history], label="Val", lw=1.5)
axes[0].axvline(result["best_epoch"], color="g", ls=":", label=f"Best ep {result['best_epoch']}")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].set_title("Loss"); axes[0].legend(fontsize=8)

# Dice
axes[1].plot(epochs, [h["avg_tissue_dice"] for h in history], label="Avg Tissue Dice", lw=1.5)
axes[1].plot(epochs, [h["avg_nuclei_dice"] for h in history], label="Avg Nuclei Dice", lw=1.5)
axes[1].plot(epochs, [h["rare_macro_dice"] for h in history], label="Rare Macro Dice", lw=1.5)
axes[1].axvline(result["best_epoch"], color="g", ls=":", label=f"Best ep {result['best_epoch']}")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Dice"); axes[1].set_title("Validation Dice"); axes[1].legend(fontsize=8)

# Selection score + branch losses
axes[2].plot(epochs, [h.get("selection_score", 0) for h in history], label="Selection score", lw=1.5)
axes[2].plot(epochs, [h.get("loss_tissue", 0) for h in history], label="Tissue loss", lw=0.8, alpha=0.6)
axes[2].plot(epochs, [h.get("loss_nc", 0) for h in history], label="NC loss", lw=0.8, alpha=0.6)
axes[2].axvline(result["best_epoch"], color="g", ls=":", label=f"Best ep {result['best_epoch']}")
axes[2].set_xlabel("Epoch"); axes[2].set_title("Selection Score"); axes[2].legend(fontsize=8)

fig.suptitle(f"Best epoch {result['best_epoch']} — score {result['best_score']:.4f}", fontsize=13)
plt.tight_layout(); plt.show()

### D. Test Set Evaluation

Metrics on the held-out test set (never used during training or checkpoint selection).

In [ ]:
test_metrics = result.get("test_metrics", {})
if test_metrics:
    tissue_names = ["stroma", "blood_vessel", "tumor", "epidermis", "necrosis"]
    nuclei_names = ["tumor", "lymphocyte", "plasma", "histiocyte", "melanophage",
                    "neutrophil", "stroma", "epithelium", "endothelium", "apoptosis"]

    print(f"{'Test Metric':25s} {'Value':>8s}")
    print("-" * 35)
    for k, v in sorted(test_metrics.items()):
        if isinstance(v, float):
            print(f"{k:25s} {v:8.4f}")

    # Per-class Dice
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for i, name in enumerate(tissue_names):
        d = test_metrics.get(f"tissue_dice_{i}", 0)
        axes[0].bar(i, d, label=f"{i}: {name}")
    axes[0].axhline(test_metrics.get("avg_tissue_dice", 0), color="r", ls="--", label=f"Avg={test_metrics.get('avg_tissue_dice',0):.4f}")
    axes[0].set_xticks(range(len(tissue_names)))
    axes[0].set_xticklabels(tissue_names, rotation=30, ha="right", fontsize=8)
    axes[0].set_ylabel("Dice"); axes[0].set_title("Tissue Per-Class Dice (Test)"); axes[0].legend(fontsize=8)

    for i, name in enumerate(nuclei_names):
        d = test_metrics.get(f"nuclei_dice_{i}", 0)
        axes[1].bar(i, d, label=f"{i}: {name}")
    axes[1].axhline(test_metrics.get("avg_nuclei_dice", 0), color="r", ls="--", label=f"Avg={test_metrics.get('avg_nuclei_dice',0):.4f}")
    axes[1].set_xticks(range(len(nuclei_names)))
    axes[1].set_xticklabels(nuclei_names, rotation=30, ha="right", fontsize=8)
    axes[1].set_ylabel("Dice"); axes[1].set_title("Nuclei Per-Class Dice (Test)"); axes[1].legend(fontsize=8)

    plt.tight_layout(); plt.show()
else:
    print("No test metrics available.")

In [ ]:
# ── Free GPU memory before prediction visualization ──
import gc
gc.collect()
torch.cuda.empty_cache()
print("GPU cache cleared before prediction viz")

### E. Prediction Visualization on Test Set

In [ ]:
from models import UnifiedPanopticNet
from models.backbone import build_cnn_backbone

# Load best checkpoint for visualization
vis_model = UnifiedPanopticNet(
    cnn_model=build_cnn_backbone(pretrained=False),
    num_tissue=5,
    num_nuclei=10,
    load_encoder_weights=False,
    fine_tune_last_n_blocks=gpu_info.fine_tune_last_n_blocks,
    use_context_encoder=gpu_info.use_context_encoder,
).to(device)
vis_model.eval()

# Build fresh 3-way dataset so we can index by position
vis_ds = PUMADataset(
    data_dir=str(PATHS.data_dir),
    transforms=get_val_transforms(C.image_size),
    context_dir=context_dir,
    use_context=C.use_context_encoder,
)

best_ckpt_path = Path("checkpoints/puma_epoch_best_s1.pth")
if best_ckpt_path.exists():
    ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)
    state = extract_state_dict(ckpt)
    vis_model.load_state_dict(state, strict=False)

    inv_mean = [-m/s for m,s in zip([0.485,0.456,0.406], [0.229,0.224,0.225])]
    inv_std  = [1/s for s in [0.229,0.224,0.225]]
    tissue_cmap = plt.cm.tab10
    nc_cmap = plt.cm.tab10

    n_show = min(4, len(test_idx))
    fig, axes = plt.subplots(n_show, 5, figsize=(20, 3 * n_show))
    if n_show == 1:
        axes = axes.reshape(1, -1)

    for row in range(n_show):
        s = vis_ds[test_idx[row]]
        img = s["image"].unsqueeze(0).to(device)
        with torch.no_grad():
            out = vis_model(img, **(
                {"context_roi": s["context_roi"].unsqueeze(0).to(device)}
                if C.use_context_encoder and "context_roi" in s else {}
            ))

        # Denormalize image
        img_np = s["image"].permute(1,2,0).numpy()
        img_np = img_np * inv_std + inv_mean
        img_np = np.clip(img_np, 0, 1)

        gt_tissue = s["tissue_sem"].numpy()
        pred_tissue = out["tissue"][0].argmax(dim=0).cpu().numpy()
        gt_nc = s["nuclei_nc"].numpy()
        pred_nc = out["nc"][0].argmax(dim=0).cpu().numpy()
        pred_np = (out["np"][0,0].sigmoid() > 0.5).cpu().numpy()

        axes[row,0].imshow(img_np)
        axes[row,0].set_title(f"Test sample {row}")
        axes[row,1].imshow(gt_tissue, cmap=tissue_cmap, vmin=0, vmax=4)
        axes[row,1].set_title("GT Tissue")
        axes[row,2].imshow(pred_tissue, cmap=tissue_cmap, vmin=0, vmax=4)
        axes[row,2].set_title("Pred Tissue")
        axes[row,3].imshow(gt_nc, cmap=nc_cmap, vmin=0, vmax=9)
        axes[row,3].set_title("GT Nuclei")
        axes[row,4].imshow(pred_nc, cmap=nc_cmap, vmin=0, vmax=9)
        axes[row,4].set_title("Pred Nuclei")
        for ax in axes[row]:
            ax.axis("off")

    plt.tight_layout(); plt.show()
else:
    print("No best checkpoint found at checkpoints/puma_epoch_best_s1.pth")

### How to use saved models for submission

Both `.pth` files are **full entity models** (architecture + weights baked in via `safe_torch_save_entity`).

| File | Usage |
|------|-------|
| `puma_epoch_best_s1.pth` | Load with `load_stage1(checkpoint, device)` in `inference/model_loader.py` |
| `puma_epoch_last_s1.pth` | Same format; use for resuming training |

**For submission**: copy the best checkpoint to your Docker container and run:
```bash
python scripts/run_inference.py --input /input --output /output --cp /app/models/puma_epoch_best_s1.pth --use-tta
```

---
## 8. Inference Demo (TTA + Post-processing)

In [ ]:
from inference.infer_wsi import apply_tta
from inference.postprocessing import hv_instance_segmentation


# TTA demo on dummy data
class DummyModel(torch.nn.Module):
    def forward(self, x, site_ids=None, context_roi=None):
        return {
            "tissue": torch.randn(1, 5, *x.shape[-2:]),
            "np": torch.randn(1, 1, *x.shape[-2:]),
            "nc": torch.randn(1, 10, *x.shape[-2:]),
            "hv": torch.randn(1, 2, *x.shape[-2:]),
            "boundary": torch.randn(1, 1, *x.shape[-2:]),
        }

dummy = DummyModel()
tensor = torch.randn(1, 3, 256, 256)
out_tta = apply_tta(dummy, tensor, site_ids=None, use_tta=True)
print(f"TTA outputs: { {k: list(v.shape) for k,v in out_tta.items()} }")

# Quick post-processing demo
dummy_np = torch.sigmoid(torch.randn(1, 1, 64, 64)).squeeze().numpy()
dummy_hv = torch.randn(2, 64, 64).numpy()
inst_map = hv_instance_segmentation(dummy_np, dummy_hv, threshold=0.5, min_size=10)
print(f"Post-processing: {len(np.unique(inst_map))-1} instances found")

In [ ]:
# ── Free GPU memory before verify exports ──
import gc
gc.collect()
torch.cuda.empty_cache()
print("GPU cache cleared")

---
## 9. Verify All Exports

In [ ]:
# ── Models ──
# ── Configs ──
from configs import PATHS
from configs.defaults import linear_ramp

# ── Data ──
from data.constants import *
from data.dataset import PUMADataset, get_train_transforms, get_val_transforms

# ── Preprocessing ──
from data.preprocessing import main as preprocess_main
from data.preprocessing.flow_generator import compute_hv_map
from data.preprocessing.geojson_parser import parse_geojson_masks

# ── Inference ──
from inference.infer_wsi import apply_tta
from inference.postprocessing import hv_instance_segmentation
from models import UnifiedPanopticNet, build_cnn_backbone
from models.components import (
    ContextEncoder,
    ContextFusionModule,
)
from models.decoders import (
    ParallelDecoders,
)
from models.fpn_aggregator import HierarchicalFPN

# ── Training ──
from training import (
    cleanup_gpu_cache,
    detect_gpu_setup,
    extract_state_dict,
    train_one_epoch,
    validate,
)
from training.stage1_trainer import apply_smooth_schedule, save_checkpoint

# ── Utils ──
from utils import MultiTaskUncertaintyLoss, PUMAMetrics, build_warmup_cosine_scheduler
from utils.scheduler_utils import build_warmup_cosine_scheduler
from utils.split_utils import make_or_load_group_split_with_test

print("All v8 imports OK.")
print("  Models: UnifiedPanopticNet, SpatialInjector, HierarchicalFPN, ParallelDecoders")
print(f"  Training: {len(__all__) if '__all__' in dir() else 'OK'} exports")
print("  Inference: infer_main, TTA, tiling, postproc, site_classifier")

In [ ]:
cleanup_gpu_cache()
print("Done.")